# Deployment Model: Train and Export

Produces the artifacts `server/` needs to run: the **60-beat causal MS-CGCA 2-way
ensemble** (XGBoost + population MS-CGCA network) under **endpoint labeling**.

Selected in `notebook-deployment-decision.ipynb` over three alternatives:

| Configuration | Macro F1 | Quadratic kappa |
| --- | --- | --- |
| BiLSTM 2-way | 0.5993 +/- 0.0120 | 0.7897 +/- 0.0232 |
| MS-CGCA 3-way | 0.5991 +/- 0.0109 | 0.7773 +/- 0.0225 |
| **MS-CGCA 2-way (this notebook)** | **0.5925 +/- 0.0129** | **0.7755 +/- 0.0174** |
| XGBoost alone | 0.5703 +/- 0.0040 | 0.7229 +/- 0.0036 |

The three sequence models are within noise of each other (all pairwise p >= 0.625).
MS-CGCA 2-way is chosen because it is causal by construction and drops the personalised
head, which added +0.0066 F1 (5-seed mean) at p = 0.625 while requiring per-user calibration state.

## What this notebook does

1. **Stage A — LOSO.** Determines the 2-way blend weight and records an honest
   generalisation estimate. This is *evaluation only*; none of these models ship.
2. **Stage B — final fit.** Retrains XGBoost, the MS-CGCA network and the scaler on
   **all 15 subjects**. This is what ships.
3. **Stage C — export and verify.** Writes the artifacts, reloads them from disk, and
   asserts the reloaded pipeline reproduces the in-memory predictions exactly.

Runtime is roughly 25-35 minutes on a T4: Stage A is a 15-fold LOSO pass, Stage B is a
single fit.

**Why Stage A cannot be skipped.** A deployed ensemble needs one weight pair, and the
endpoint-labeled 2-way weight was never recorded — the decision notebook selected weights
nested per fold and discarded them. Stage A recovers it, and reports both the pooled
figure (optimistic: selected and scored on the same windows) and the nested figure
(unbiased). Only the nested figure goes into `model_config.json`.

## 1. Setup

In [1]:
!pip install neurokit2 xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 15.7 MB/s eta 0:00:00


In [2]:
import os, pickle, warnings, json, time, hashlib
import numpy as np
from scipy.signal import welch
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_PATH = '/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE = '/kaggle/working'
ART  = f'{SAVE}/artifacts'
for sub in ['models', 'scalers', 'config', 'fixtures']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)

SUBJECT_IDS = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES = ['relaxed','mild','moderate','high']; NCLS = 4

WINDOW = 60
STEP   = 5
LABELING = 'endpoint'                     # y = labels[e-1]
EWMA_HALFLIVES = {'fast':60, 'medium':300, 'slow':1800}
POPULATION_RR_MS = 780.0
ROLL_WINDOW = 20
ZSCORE_HALFLIFE = 300

XGB_FEATURE_ORDER = [
    'mean_RR','SDNN','RMSSD','pNN50','CV_RR','VLF','LF','HF','LF/HF','LF_nu',
    'SD1','SD2','SD1/SD2',
    'res_mean','res_SD','res_maxabs','res_slope','res_msq',
    'ewma_fast_level','ewma_slow_level',
    'sin_24h','cos_24h','sin_90m','cos_90m','cortisol',
]
CNN_SEQUENCE_CHANNELS = ['rn','rm','sd','hr','rrn','tn','trn']

print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU')) > 0)
print("artifacts ->", ART)

TF 2.20.0 GPU True
artifacts -> /kaggle/working/artifacts


## 2. Pipeline

Transcribed from `notebook-newmodel.ipynb` and re-verified in
`notebook-deployment-decision.ipynb`. `src/componentb/` must match these functions
exactly — any divergence is training-serving skew.

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl", 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg = nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs); rp = info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr, ts):
    rr = rr.copy().astype(float); rr[(rr <= 300) | (rr >= 2000)] = np.nan
    for i in range(1, len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1] > 0.20: rr[i] = np.nan
    m = np.isnan(rr)
    if m.any(): rr[m] = np.interp(np.where(m)[0], np.where(~m)[0], rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap = np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out = []
    for i in range(len(rp)-1):
        seg = labels[rp[i]:rp[i+1]]; v = seg[seg > 0]
        out.append(0 if len(v) == 0 else np.bincount(v).argmax())
    return np.array(out)

# ---- causal replacements (past-only by construction) ----
def ewma_causal(x, halflife):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    o = np.empty(len(x), dtype=float); state = float(POPULATION_RR_MS)
    for i in range(len(x)):
        state = a*x[i] + (1-a)*state; o[i] = state
    return o

def causal_zscore(x, halflife=ZSCORE_HALFLIFE):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    mu = np.empty(len(x)); sd = np.empty(len(x))
    m = float(x[0]) if len(x) else 0.0; v = 1.0
    for i in range(len(x)):
        d = x[i]-m; m = m + a*d; v = (1-a)*(v + a*d*d)
        mu[i] = m; sd[i] = np.sqrt(max(v, 1e-8))
    return (x-mu)/(sd+1e-8)

def roll_rmssd_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.sqrt(np.mean(np.diff(seg)**2)) if len(seg) > 1 else 0.0
    return o

def roll_sdnn_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.std(seg) if len(seg) > 1 else 0.0
    return o

def hrv_features(w, fs=4.0):
    rr, diff = np.array(w), np.diff(w)
    mean_rr = np.mean(rr); sdnn = np.std(rr); rmssd = np.sqrt(np.mean(diff**2))
    pnn50 = np.sum(np.abs(diff) > 50)/len(diff)*100; cv = sdnn/mean_rr
    t = np.cumsum(rr)/1000.0; u = np.interp(np.arange(0, t[-1], 1/fs), t, rr)
    fr, psd = welch(u, fs=fs, nperseg=min(256, len(u)))
    vlf = TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf = TRAPZ(psd[(fr>=0.04)&(fr<0.15)])
    hf  = TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf = lf/(hf+1e-8); lf_nu = lf/(lf+hf+1e-8)
    sd1 = np.sqrt(0.5)*np.std(diff); sd2 = np.sqrt(max(2*sdnn**2 - 0.5*np.var(diff), 0))
    return np.array([mean_rr, sdnn, rmssd, pnn50, cv, vlf, lf, hf, lf_hf, lf_nu,
                     sd1, sd2, sd1/(sd2+1e-8)])

def resid_features(rw):
    r = np.array(rw)
    return np.array([np.mean(r), np.std(r), np.max(np.abs(r)),
                     np.polyfit(np.arange(len(r)), r, 1)[0], np.sum(r**2)/len(r)])

def circ_features(ts):
    t, hour = ts % 86400, (ts % 86400)/3600.0
    cort = 0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400), cort])

def circ7(ts):
    t = ts % 86400; hour = t/3600.0
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400),
                     0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                     np.sin(2*np.pi*(hour-23)/24), np.cos(2*np.pi*(hour-23)/24)])

print("pipeline defined")

pipeline defined


### Build windows (endpoint labeling)

The label index is `e-1`, the window's last beat, and the time-of-day feature index
follows it. Nothing in the feature vector postdates the labelled beat.

In [4]:
raw = {}
for sid in SUBJECT_IDS:
    chest, wt, labels = load_subject(sid); ecg = chest['ECG'].flatten()
    rr, ts, rp = extract_rr_from_ecg(ecg); temp = align_temp(wt, rp)
    rr, ts = clean_rr(rr, ts)
    rl = labels_to_rr(labels, rp); keep = rl > 0
    rrk, tk, tsk, lk = rr[keep], temp[keep], ts[keep], rl[keep]
    new = np.zeros(len(lk), dtype=int); si = np.where(lk == 2)[0]
    if len(si) > 0:
        srr = rrk[si]; loc = []
        for i in range(len(srr)):
            w = srr[max(0, i-15):i+15]; dd = np.diff(w)
            loc.append(np.sqrt(np.mean(dd**2)) if len(dd) > 0 else 50)
        loc = np.array(loc); p33, p66 = np.percentile(loc, 33), np.percentile(loc, 66)
        for i, idx in enumerate(si):
            new[idx] = (1 if loc[i] >= p66 else 2 if loc[i] >= p33 else 3)
    raw[sid] = dict(rr=rrk, temp=tk, ts=tsk, lab=new)
print(len(raw), "subjects"); assert len(raw) == 15

def build_endpoint():
    Xs, Xc, Xx, y, g = [], [], [], [], []
    for sid, d in raw.items():
        rr, temp, ts, labels = d['rr'], d['temp'], d['ts'], d['lab']
        base = {k: ewma_causal(rr, hl) for k, hl in EWMA_HALFLIVES.items()}
        res_med  = rr - base['medium']
        temp_res = temp - ewma_causal(temp, EWMA_HALFLIVES['medium'])
        rn  = causal_zscore(rr); rm = roll_rmssd_causal(rn); sd = roll_sdnn_causal(rn)
        hr  = 60000/(rr+1e-8); rrn = causal_zscore(res_med)
        tn  = causal_zscore(temp); trn = causal_zscore(temp_res)
        for s in range(0, len(rr)-WINDOW, STEP):
            e  = s + WINDOW
            li = e - 1                       # endpoint label
            bi = min(li, len(ts)-1)
            seq = np.stack([rn[s:e], rm[s:e], sd[s:e], hr[s:e],
                            rrn[s:e], tn[s:e], trn[s:e]], axis=-1)
            try:
                xf = np.concatenate([hrv_features(rr[s:e]), resid_features(res_med[s:e]),
                                     np.array([base['fast'][e-1], base['slow'][e-1]]),
                                     circ_features(ts[bi])])
            except Exception:
                continue
            Xs.append(seq); Xc.append(circ7(ts[bi])); Xx.append(xf)
            y.append(labels[li]); g.append(sid)
    return (np.array(Xs, np.float32), np.array(Xc, np.float32), np.array(Xx),
            np.array(y, np.int32), np.array(g, np.int32))

X_seq, X_circ, X_xgb, y_all, groups = build_endpoint()
print("seq", X_seq.shape, "circ", X_circ.shape, "xgb", X_xgb.shape)
print("classes", np.bincount(y_all, minlength=4))
assert X_xgb.shape[1] == len(XGB_FEATURE_ORDER), "feature count != declared order"
assert X_seq.shape[2] == len(CNN_SEQUENCE_CHANNELS)

15 subjects
seq (12026, 60, 7) circ (12026, 7) xgb (12026, 25)
classes [8775 1099 1085 1067]


## 3. Architecture

In [5]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def call(self, yt, yp):
        yt = tf.cast(yt, tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(yt, yp)
        pt = tf.reduce_sum(tf.one_hot(yt, 4)*yp, axis=-1)
        return tf.pow(1.0-pt, self.gamma)*ce

def build_ms_cgca(window=WINDOW, nch=7, ncirc=7, ncls=4):
    si = layers.Input(shape=(window, nch), name='sequence')
    ci = layers.Input(shape=(ncirc,), name='circadian')
    c1 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=1)(si)
    c2 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=2)(si)
    c4 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=4)(si)
    x = layers.Concatenate()([c1, c2, c4])
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.4)(x)
    cp = layers.Dense(128, activation='relu')(ci)
    cp = layers.RepeatVector(window//2)(cp)
    attn = layers.Attention()([cp, x])
    x = layers.GlobalAveragePooling1D()(attn)
    cf = layers.Dense(32, activation='relu')(ci)
    m = layers.Concatenate()([x, cf])
    o = layers.Dense(64, activation='relu')(m); o = layers.Dropout(0.4)(o)
    return Model([si, ci], layers.Dense(ncls, activation='softmax')(o), name='MS_CGCA')

def build_bilstm(window=WINDOW, nch=7, ncirc=7, ncls=4):
    si = layers.Input(shape=(window, nch), name='sequence')
    ci = layers.Input(shape=(ncirc,), name='circadian')
    x = layers.Conv1D(64, 7, padding='same', activation='relu')(si)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.4)(x); a = layers.Attention()([x, x])
    x = layers.GlobalAveragePooling1D()(a)
    c = layers.Dense(32, activation='relu')(ci); c = layers.Dense(16, activation='relu')(c)
    x = layers.Concatenate()([x, c]); x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    return Model([si, ci], layers.Dense(ncls, activation='softmax')(x), name='BiLSTM')

def make_xgb(seed):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, objective='multi:softprob', num_class=4,
                         eval_metric='mlogloss', random_state=seed, n_jobs=-1)

def freeze_extractor(m):
    for l in m.layers:
        ln = l.__class__.__name__.lower()
        l.trainable = not any(k in ln for k in ['conv','lstm','batchnorm','attention','pooling'])
    return m

def strat_calib(sub_idx, y, frac=0.2, minpc=5):
    lab = y[sub_idx]; ntot = int(len(sub_idx)*frac); calib = []
    rng = np.random.RandomState(42)          # fixed: the split must not vary with seed
    for c in np.unique(lab):
        pos = sub_idx[lab == c]
        take = min(max(minpc, ntot//len(np.unique(lab))), len(pos))
        calib.extend(rng.choice(pos, take, replace=False))
    calib = np.array(sorted(calib))
    ev = np.array([i for i in sub_idx if i not in set(calib)])
    return calib, ev

print("architectures defined")

architectures defined


## 4. Stage A — LOSO: select the blend weight, record honest metrics

Each fold trains on 14 subjects and predicts the 15th. Two weights are then derived:

- **pooled** — the weight maximising F1 over all folds' predictions at once. Selected and
  scored on the same windows, so its score is optimistic.
- **nested** — for each held-out subject, the weight chosen on the other fourteen only.
  Never scores a subject with a weight its own data influenced.

The pooled *weight* is what ships (it is the single best estimate of the right blend);
the nested *score* is what gets recorded as the model's expected performance.

In [6]:
def macro_f1(yt, yp, K=4):
    f = []
    for c in range(K):
        tp = np.sum((yp==c)&(yt==c)); fp = np.sum((yp==c)&(yt!=c)); fn = np.sum((yp!=c)&(yt==c))
        f.append(0.0 if tp == 0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))

WGRID = [(round(w,2), round(1-w,2)) for w in np.arange(0.10, 0.91, 0.05)]   # (w_xgb, w_cnn)

CKPT = f'{SAVE}/loso_folds.npz'
folds = []
if os.path.exists(CKPT):
    z = np.load(CKPT, allow_pickle=True); folds = list(z['folds'])
    print(f"loaded {len(folds)} cached folds")

t0 = time.time()
for i, (tr, te) in enumerate(LeaveOneGroupOut().split(X_xgb, y_all, groups)):
    if i < len(folds):
        continue
    sc = StandardScaler().fit(X_xgb[tr])
    xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, objective='multi:softprob', num_class=4,
                        eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
    xgb.fit(sc.transform(X_xgb[tr]), y_all[tr],
            sample_weight=compute_sample_weight('balanced', y_all[tr]), verbose=False)
    p_xgb = xgb.predict_proba(sc.transform(X_xgb[te]))

    cw  = compute_class_weight('balanced', classes=np.unique(y_all[tr]), y=y_all[tr])
    net = build_ms_cgca()
    net.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=SparseFocalLoss(2.0))
    net.fit([X_seq[tr], X_circ[tr]], y_all[tr], validation_split=0.15, epochs=120,
            batch_size=32, class_weight=dict(enumerate(cw)),
            callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                               restore_best_weights=True),
                       callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7)],
            verbose=0)
    p_cnn = net.predict([X_seq[te], X_circ[te]], verbose=0)

    folds.append(dict(y=y_all[te], p_xgb=p_xgb, p_cnn=p_cnn))
    np.savez_compressed(CKPT, folds=np.array(folds, dtype=object))
    print(f"fold {i+1:2d}/15  S{int(np.unique(groups[te])[0]):02d}  n={len(te):5d}  ({(time.time()-t0)/60:.1f} min)")
    tf.keras.backend.clear_session()

print(f"\nStage A complete: {(time.time()-t0)/60:.1f} min")

I0000 00:00:1787114177.982710      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787114177.985969      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


fold  1/15  S02  n=  707  (1.2 min)
fold  2/15  S03  n=  624  (2.2 min)
fold  3/15  S04  n=  668  (3.7 min)
fold  4/15  S05  n=  727  (4.8 min)
fold  5/15  S06  n=  731  (6.2 min)
fold  6/15  S07  n=  714  (7.3 min)
fold  7/15  S08  n=  776  (8.6 min)
fold  8/15  S09  n=  810  (10.8 min)
fold  9/15  S10  n=  977  (11.9 min)
fold 10/15  S11  n=  926  (13.2 min)
fold 11/15  S13  n=  916  (14.7 min)
fold 12/15  S14  n=  926  (15.9 min)
fold 13/15  S15  n=  824  (17.3 min)
fold 14/15  S16  n=  885  (18.9 min)
fold 15/15  S17  n=  815  (21.7 min)

Stage A complete: 21.7 min


In [7]:
allY = np.concatenate([f['y'] for f in folds])

def pooled_score(w):
    wx, wc = w
    preds = np.concatenate([(wx*f['p_xgb'] + wc*f['p_cnn']).argmax(1) for f in folds])
    return macro_f1(allY, preds)

W_POOLED = max(WGRID, key=pooled_score)
wx, wc = W_POOLED
pooled_preds = np.concatenate([(wx*f['p_xgb'] + wc*f['p_cnn']).argmax(1) for f in folds])
pooled_f1 = macro_f1(allY, pooled_preds)
pooled_k  = float(cohen_kappa_score(allY, pooled_preds, weights='quadratic'))

nested_preds = []
for i, fS in enumerate(folds):
    inner  = [f for j, f in enumerate(folds) if j != i]
    innerY = np.concatenate([f['y'] for f in inner])
    def inner_f1(w):
        a, b = w
        return macro_f1(innerY, np.concatenate([(a*f['p_xgb'] + b*f['p_cnn']).argmax(1) for f in inner]))
    a, b = max(WGRID, key=inner_f1)
    nested_preds.append((a*fS['p_xgb'] + b*fS['p_cnn']).argmax(1))
nested_preds = np.concatenate(nested_preds)
nested_f1 = macro_f1(allY, nested_preds)
nested_k  = float(cohen_kappa_score(allY, nested_preds, weights='quadratic'))

print("="*68)
print(f"shipped blend weight   w_xgb={wx:.2f}  w_cnn={wc:.2f}")
print("-"*68)
print(f"pooled (optimistic)    F1={pooled_f1:.4f}  kappa={pooled_k:.4f}")
print(f"nested (unbiased)      F1={nested_f1:.4f}  kappa={nested_k:.4f}   <- record this")
print(f"selection bias                {pooled_f1-nested_f1:+.4f}")
print("-"*68)
print(f"accuracy               {np.mean(allY==nested_preds):.4f}")
print(f"severe errors (|e|>=2) {np.mean(np.abs(allY-nested_preds)>=2):.4f}")
print(f"within-1 accuracy      {np.mean(np.abs(allY-nested_preds)<=1):.4f}")
print(f"eval windows           {len(allY)}")
print("="*68)
print("\nFor comparison, the 5-seed decision-notebook estimate for this configuration")
print("was F1 = 0.5925 +/- 0.0129, kappa = 0.7755 +/- 0.0174. This single-seed run")
print("should fall inside roughly two standard deviations of that; if it does not,")
print("something in the pipeline has drifted and the export should not proceed.")

shipped blend weight   w_xgb=0.20  w_cnn=0.80
--------------------------------------------------------------------
pooled (optimistic)    F1=0.5970  kappa=0.7811
nested (unbiased)      F1=0.5970  kappa=0.7811   <- record this
selection bias                +0.0000
--------------------------------------------------------------------
accuracy               0.8354
severe errors (|e|>=2) 0.0535
within-1 accuracy      0.9465
eval windows           12026

For comparison, the 5-seed decision-notebook estimate for this configuration
was F1 = 0.5925 +/- 0.0129, kappa = 0.7755 +/- 0.0174. This single-seed run
should fall inside roughly two standard deviations of that; if it does not,
something in the pipeline has drifted and the export should not proceed.


## 5. Stage B — final fit on all 15 subjects

Nothing is held out here. The scaler, XGBoost model and network are fit on the complete
dataset, because at deployment there is no held-out subject — every user is unseen. The
Stage A numbers remain the honest estimate of how this performs on someone new.

In [8]:
t0 = time.time()

final_scaler = StandardScaler().fit(X_xgb)
Xs_scaled = final_scaler.transform(X_xgb)

final_xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, objective='multi:softprob', num_class=4,
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
final_xgb.fit(Xs_scaled, y_all,
              sample_weight=compute_sample_weight('balanced', y_all), verbose=False)

cw = compute_class_weight('balanced', classes=np.unique(y_all), y=y_all)
final_net = build_ms_cgca()
final_net.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=SparseFocalLoss(2.0))
hist = final_net.fit([X_seq, X_circ], y_all, validation_split=0.15, epochs=120,
                     batch_size=32, class_weight=dict(enumerate(cw)),
                     callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                                        restore_best_weights=True),
                                callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                            patience=7)],
                     verbose=0)
print(f"final fit complete: {(time.time()-t0)/60:.1f} min, {len(hist.history['loss'])} epochs")

# in-sample only -- NOT a performance estimate, just a sanity check that fitting worked
p = (wx*final_xgb.predict_proba(Xs_scaled) + wc*final_net.predict([X_seq, X_circ], verbose=0))
print(f"in-sample F1 {macro_f1(y_all, p.argmax(1)):.4f}  (expected to be high; not a generalisation estimate)")

final fit complete: 1.9 min, 33 epochs
in-sample F1 0.8895  (expected to be high; not a generalisation estimate)


## 6. Stage C — export

In [9]:
final_net.save(f'{ART}/models/mscgca_population.keras')
final_xgb.save_model(f'{ART}/models/xgb_population.json')
with open(f'{ART}/scalers/feature_scaler.pkl', 'wb') as f:
    pickle.dump(final_scaler, f)

model_config = {
    "model_type": "two-way causal MS-CGCA ensemble",
    "window_beats": WINDOW,
    "step_beats": STEP,
    "labeling": "endpoint",
    "labeling_note": "y = labels[e-1], the window's last beat. Midpoint labeling inflated "
                     "macro-F1 by +0.071 to +0.084 across all configurations and is not deployable.",
    "ensemble_weights": {"xgb": float(wx), "cnn": float(wc)},
    "ewma_halflives": EWMA_HALFLIVES,
    "zscore_halflife": ZSCORE_HALFLIFE,
    "population_rr_ms": POPULATION_RR_MS,
    "roll_window": ROLL_WINDOW,
    "xgb_feature_order": XGB_FEATURE_ORDER,
    "xgb_feature_dim": int(X_xgb.shape[1]),
    "cnn_sequence_channels": CNN_SEQUENCE_CHANNELS,
    "cnn_circadian_dim": int(X_circ.shape[1]),
    "class_names": CLASS_NAMES,
    "n_classes": NCLS,
    "loso_macro_f1": round(nested_f1, 4),
    "loso_kappa": round(nested_k, 4),
    "loso_macro_f1_pooled": round(pooled_f1, 4),
    "loso_kappa_pooled": round(pooled_k, 4),
    "eval_windows": int(len(allY)),
    "total_windows": int(len(y_all)),
    "seed": SEED,
    "trained_on": "all 15 WESAD subjects (no holdout); LOSO metrics above are the "
                  "generalisation estimate for an unseen user",
    "source_notebook": "notebook-train-export-2way.ipynb",
    "note": "Supersedes the 120-beat two-way ensemble (F1 0.5962, kappa 0.7858) and the "
            "3-way MS-CGCA ensemble. The personalised head was dropped: +0.0066 F1 (5-seed mean) at "
            "Wilcoxon p = 0.625, not distinguishable from noise. loso_macro_f1 is the "
            "nested, leakage-free figure; the pooled figure is recorded alongside it for "
            "transparency only and must not be quoted as performance.",
}
with open(f'{ART}/config/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

# parity fixture: 200 windows + expected probabilities, for tests/test_parity.py
idx = np.random.RandomState(0).choice(len(y_all), 200, replace=False)
np.savez_compressed(f'{ART}/fixtures/parity_fixture.npz',
                    X_xgb=X_xgb[idx], X_seq=X_seq[idx], X_circ=X_circ[idx],
                    y=y_all[idx],
                    p_xgb=final_xgb.predict_proba(final_scaler.transform(X_xgb[idx])),
                    p_cnn=final_net.predict([X_seq[idx], X_circ[idx]], verbose=0))

for root, _d, files in os.walk(ART):
    for fn in sorted(files):
        p_ = os.path.join(root, fn)
        print(f"  {p_.replace(ART+'/',''):45s} {os.path.getsize(p_)/1024:8.1f} KB")

  scalers/feature_scaler.pkl                         1.0 KB
  fixtures/parity_fixture.npz                      332.8 KB
  config/model_config.json                           1.8 KB
  models/mscgca_population.keras                  1601.0 KB
  models/xgb_population.json                      5019.1 KB


### Verify the export

Reload every artifact from disk and assert the reloaded pipeline reproduces the
in-memory predictions. A silently mismatched scaler produces no error at inference — it
just degrades predictions — so this check is the only thing standing between a bad export
and a quiet failure in production.

In [10]:
with open(f'{ART}/scalers/feature_scaler.pkl','rb') as f:
    sc2 = pickle.load(f)
xgb2 = XGBClassifier(); xgb2.load_model(f'{ART}/models/xgb_population.json')
net2 = tf.keras.models.load_model(f'{ART}/models/mscgca_population.keras', compile=False)
cfg2 = json.load(open(f'{ART}/config/model_config.json'))
fx   = np.load(f'{ART}/fixtures/parity_fixture.npz')

assert np.allclose(sc2.mean_, final_scaler.mean_) and np.allclose(sc2.scale_, final_scaler.scale_)
p_xgb2 = xgb2.predict_proba(sc2.transform(fx['X_xgb']))
p_cnn2 = net2.predict([fx['X_seq'], fx['X_circ']], verbose=0)
assert np.allclose(p_xgb2, fx['p_xgb'], atol=1e-5), "XGBoost mismatch after reload"
assert np.allclose(p_cnn2, fx['p_cnn'], atol=1e-4), "network mismatch after reload"

w = cfg2['ensemble_weights']
blend_disk = w['xgb']*p_xgb2 + w['cnn']*p_cnn2
blend_mem  = wx*fx['p_xgb'] + wc*fx['p_cnn']
assert np.array_equal(blend_disk.argmax(1), blend_mem.argmax(1)), "blend mismatch"
assert abs(w['xgb'] + w['cnn'] - 1.0) < 1e-9
assert cfg2['window_beats'] == WINDOW and cfg2['labeling'] == 'endpoint'
assert cfg2['xgb_feature_dim'] == X_xgb.shape[1] == len(cfg2['xgb_feature_order'])

print("all export checks passed")
print(f"  window {cfg2['window_beats']} beats, labeling {cfg2['labeling']}")
print(f"  weights xgb={w['xgb']:.2f} cnn={w['cnn']:.2f}")
print(f"  LOSO (nested) F1={cfg2['loso_macro_f1']:.4f} kappa={cfg2['loso_kappa']:.4f}")

print("\nSHA-256 (record these in docs/ARCHITECTURE.md so the files can be identified later):")
for rel in ['models/mscgca_population.keras','models/xgb_population.json',
            'scalers/feature_scaler.pkl','config/model_config.json']:
    h = hashlib.sha256(open(f'{ART}/{rel}','rb').read()).hexdigest()
    print(f"  {rel:38s} {h[:16]}")

all export checks passed
  window 60 beats, labeling endpoint
  weights xgb=0.20 cnn=0.80
  LOSO (nested) F1=0.5970 kappa=0.7811

SHA-256 (record these in docs/ARCHITECTURE.md so the files can be identified later):
  models/mscgca_population.keras         1c6d84fd0af0c1d5
  models/xgb_population.json             2a801f18dd6a4b47
  scalers/feature_scaler.pkl             95dcfe74685280f9
  config/model_config.json               f396b0407edaaa9e


## 7. After this notebook

1. Download `artifacts/` and place it at the repo root. It is gitignored — the files are
   distributed separately, not committed.
2. Update `src/componentb/config.py`: `WINDOW_BEATS = 60`, and add
   `ZSCORE_HALFLIFE = 300`. The 120-beat constant belongs to the superseded pipeline.
3. `features/hrv.py` currently produces 13 + 5 = 18 features. The exported scaler expects
   **25**: the two EWMA baseline levels and the five circadian encodings are missing.
   `causal_zscore` and `roll_rmssd_causal` are also referenced by the docs but absent from
   `src/`. These must be added, in the order given by `xgb_feature_order`, before the
   server can load the scaler.
4. `models/loader.py` loads a two-model ensemble — correct for this export. Remove any
   third-member handling; the personalised head is not shipping.
5. Point `tests/test_parity.py` at `artifacts/fixtures/parity_fixture.npz` and assert that
   `StreamingInference` reproduces `p_xgb` and `p_cnn` for those 200 windows.

**Then update the docs.** `README.md` and `docs/ARCHITECTURE.md` still carry
midpoint-labeled figures (F1 0.6708-0.6825, kappa 0.8386-0.8497, accuracy 91.69-91.93%).
Replace them with `loso_macro_f1` and `loso_kappa` from the exported config. The
accuracy figure has no endpoint-labeled equivalent yet and should be removed rather than
carried over.

**One caveat worth stating plainly in the docs.** These metrics come from WESAD chest ECG.
The deployment reads wrist PPG, and direct WESAD-to-Empatica transfer was measured at
F1 = 0.135 — a collapse. Nothing here establishes that the shipped model performs at
F1 ~ 0.59 on the actual hardware; that requires either PPG-domain training data or a
measured domain-adaptation step, and remains the largest open risk in the deployment.